# Fase 02 — Shared splits and evaluation protocol

This notebook only freezes and verifies split assignments. It does not compute descriptors or train models.

- Data are reloaded from the original MongoDB collections and cleaned by `load_cohort`.
- Each sample belongs to exactly one of five shuffled outer test folds, independently for each cohort.
- Inner folds and early-stopping validation are fixed inside the outer training subset.
- JSON manifests contain IDs, data fingerprints and assignments, **not** training inputs.
- A changed cohort or MD input triggers an error; existing manifests are never overwritten.

The three Tg comparisons are independent. For other properties only one cleaned source cohort per target is used. Random splits measure performance within the available datasets, not extrapolation to unseen polymer families.

In [1]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/final_comparison/datasets.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the polymer_prediction repository")
sys.path.insert(0, str(ROOT))

from src.final_comparison.datasets import COLLECTIONS, DB_NAME, MD_COLUMNS, load_cohort
from src.final_comparison.splits import (
    SEED, create_manifest, file_sha256, fold_summary,
    load_manifest, save_manifest,
)
from runlog import RunLogger
import pandas as pd

In [2]:
MD_DIR = ROOT / "data/md_features"
SPLIT_DIR = ROOT / "data/splits/v1"

# No target-specific cohorts are shared across different properties.
COHORTS = [
    ("Tg", "md_common", "bicerano", "K"),
    ("Tg", "bicerano", "bicerano", "K"),
    ("Tg", "polyinfo", "polyinfo", "K"),
    ("density", "extended", "extended", None),
    ("Tc", "extended", "extended", None),
    ("glass_cte", "bicerano", "bicerano", "10^-6/K"),
    ("rubber_cte", "bicerano", "bicerano", "10^-6/K"),
]

md_paths = {name: MD_DIR / name for name in MD_COLUMNS}
md_hashes = {name: file_sha256(path) for name, path in md_paths.items()}
print(f"Seed: {SEED} | Manifest directory: {SPLIT_DIR}")

Seed: 28 | Manifest directory: C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\data\splits\v1


In [3]:
summary = []
fold_tables = []

for target, cohort, source, unit in COHORTS:
    use_md = cohort == "md_common"
    frame, excluded = load_cohort(
        source, target=target, md_dir=MD_DIR if use_md else None,
    )
    source_collection = COLLECTIONS[source]
    manifest = create_manifest(
        frame, cohort=cohort, target=target, target_unit=unit,
        source={"db": DB_NAME, "collection": source_collection},
        source_hashes=md_hashes if use_md else {},
    )
    path = SPLIT_DIR / f"{target}_{cohort}.json"
    created = save_manifest(path, manifest)
    load_manifest(path, frame=frame, source_paths=md_paths if use_md else None)

    summary.append({
        "target": target, "cohort": cohort, "n": len(frame),
        "excluded": len(excluded), "path": str(path.relative_to(ROOT)),
        "sha256": file_sha256(path), "created": created,
    })
    fold_tables.append(fold_summary(manifest))
    print(f"{target:11s} {cohort:10s}: N={len(frame):5d} "
          f"| {'created' if created else 'verified'} | {path.name}")

summary_df = pd.DataFrame(summary)
fold_df = pd.concat(fold_tables, ignore_index=True)
display(summary_df)
display(fold_df)

Tg          md_common : N=  135 | created | Tg_md_common.json
Tg          bicerano  : N=  312 | created | Tg_bicerano.json
Tg          polyinfo  : N= 7077 | created | Tg_polyinfo.json
density     extended  : N= 1068 | created | density_extended.json
Tc          extended  : N= 1068 | created | Tc_extended.json
glass_cte   bicerano  : N=  221 | created | glass_cte_bicerano.json
rubber_cte  bicerano  : N=  312 | created | rubber_cte_bicerano.json


,target,cohort,n,excluded,path,sha256,created
0,Tg,md_common,135,180,data\splits\v1\Tg_md_common.json,b5c05377824f071fdce4f8b0197084fde969cd3aaefedb...,True
1,Tg,bicerano,312,3,data\splits\v1\Tg_bicerano.json,ce842f7b240339cb5c6195817d75a8f75b81f9d2f7c7c7...,True
2,Tg,polyinfo,7077,207,data\splits\v1\Tg_polyinfo.json,3981872cbd13e6161a18443e49418f90f6ee4ff25159d2...,True
3,density,extended,1068,20,data\splits\v1\density_extended.json,8158f257df997f4db0e4f17c118ac618696ea823a21356...,True
4,Tc,extended,1068,20,data\splits\v1\Tc_extended.json,052d20e7024c7e414fd953f036322f94b179404b654f96...,True
5,glass_cte,bicerano,221,94,data\splits\v1\glass_cte_bicerano.json,b495010432e86e6c23376f5f068c332c23a57640f38fe9...,True
6,rubber_cte,bicerano,312,3,data\splits\v1\rubber_cte_bicerano.json,90d0404fbb43d70679037273595db4809dc252bd51d1ea...,True


,cohort,target,fold,train,early_stop_train,early_stop_val,test
0,md_common,Tg,1,108,94,14,27
1,md_common,Tg,2,108,94,14,27
2,md_common,Tg,3,108,94,14,27
3,md_common,Tg,4,108,94,14,27
4,md_common,Tg,5,108,94,14,27
5,bicerano,Tg,1,249,217,32,63
6,bicerano,Tg,2,249,217,32,63
7,bicerano,Tg,3,250,218,32,62
8,bicerano,Tg,4,250,218,32,62
9,bicerano,Tg,5,250,218,32,62


In [4]:
log = RunLogger(
    config={
        "db": DB_NAME, "collection": "shared_splits", "target": "multiple",
        "phase": "02", "random_state": SEED, "use_md": False,
        "outer_folds": 5, "inner_folds": 5,
    },
    tag="phase02-freeze",
)
log.result("manifests", {r["path"]: {
    "target": r["target"], "cohort": r["cohort"], "n": r["n"],
    "sha256": r["sha256"], "created": r["created"],
} for r in summary})
log.table("split_cohorts", summary_df, index=False)
log.table("split_fold_sizes", fold_df, index=False)
log.note("Outer folds are shared within cohorts, not across separate cohorts.")
log.note("Inner CV and early-stopping validation are subsets of each outer train.")
log.finish(quiet=True)
print(f"Audit logged to {log.dir}")

[runlog] C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runs\11-20_1310_shared_splits_nomd_phase02-freeze
Audit logged to C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runs\11-20_1310_shared_splits_nomd_phase02-freeze


C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runlog.py:377: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_csv(index_path), df], ignore_index=True)


## Use in the next phases (not executed here)

Every model reloads the cleaned MongoDB cohort, calls `load_manifest(path, frame=cleaned)` and uses `fold_ids(manifest, fold)` to retrieve ID-based assignments. For nested CV, the inner training IDs are `outer_train_ids - inner_validation_ids`. The saved `early_stop_train_ids` and `early_stop_validation_ids` can be used when fitting the final GNN for an outer fold. If a GNN is tuned using inner CV, its early-stopping split must instead be drawn from each inner training fold. The outer test IDs are never used for tuning or checkpoint selection.

If a representation cannot encode every sample, preserve the frozen outer assignments while defining and documenting a separate representable subset. Retrain the comparator baselines on **that exact subset**, without equating its results to those on the full cohort.